# SelvaSonic — Análisis de Errores del Modelo Attention (Semana 5)

## Objetivo

Replicar el análisis de errores del notebook 04 (baseline) sobre el modelo de attention. Mismas preguntas, misma metodología:

> **¿Qué confunde el attention? ¿Por qué?**

El valor real de este análisis aparece cuando se contrasta con el baseline: **¿el attention atacó las confusiones que esperábamos?** Esa comparación se hace en el notebook 09.

## Estructura (idéntica al notebook 04)

| Sección | Pregunta |
|---|---|
| 1. Setup + predicciones | Reconstruir test set + cargar attention |
| 2. Matriz de confusión normalizada | Patrón de confusiones |
| 3. F1 vs cantidad de datos | ¿Sigue dominando el desbalance? |
| 4. Errores de alta confianza | Top fallos "vergonzosos" |
| 5. Confusiones bioacústicas | Clasificación taxonómica |
| 6. Calibración / umbral | Confianza aciertos vs errores |
| 7. Resumen | Hallazgos clave para comparar con baseline |

## Sección 1 — Setup y predicciones

In [ ]:
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / 'src').exists():
    sys.path.insert(0, str(PROJECT_ROOT))
elif (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from src.dataset import create_dataloaders
from src.model import SelvaSonicCNNAttention

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

COLOR_PRIMARY = '#6C5CE7'
COLOR_ACCENT = '#00CEC9'
COLOR_WARN = '#FD79A8'
COLOR_DARK = '#2D3436'

RUN_DIR = PROJECT_ROOT / 'results' / 'runs' / 'attention_S4_v1_20260601_0334'
BEST_CKPT = RUN_DIR / 'best.pth'
RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'raw'

assert BEST_CKPT.exists(), f'Falta {BEST_CKPT}'
print(f'Run: {RUN_DIR.name}')

In [ ]:
_, _, test_loader, label_map = create_dataloaders(
    raw_data_dir=str(RAW_DATA_DIR),
    batch_size=32, num_workers=0,
    train_ratio=0.70, val_ratio=0.15, test_ratio=0.15,
    random_state=42, verbose=False,
)
NUM_CLASSES = len(label_map)
idx_to_name = {v: k for k, v in label_map.items()}
class_names = [idx_to_name[i] for i in range(NUM_CLASSES)]

ckpt = torch.load(BEST_CKPT, map_location=device)
model = SelvaSonicCNNAttention(num_classes=NUM_CLASSES).to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

all_true, all_pred, all_probs = [], [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        probs = F.softmax(model(x), dim=1)
        all_true.append(y.cpu())
        all_pred.append(probs.argmax(dim=1).cpu())
        all_probs.append(probs.cpu())

y_true = torch.cat(all_true).numpy()
y_pred = torch.cat(all_pred).numpy()
y_probs = torch.cat(all_probs).numpy()
y_conf = y_probs.max(axis=1)

print(f'Accuracy global Attention: {(y_true == y_pred).mean():.4f}')
print(f'Accuracy global Baseline (ref): 0.6322')

## Sección 2 — Matriz de confusión normalizada por fila

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
with np.errstate(all='ignore'):
    cm_norm = cm / cm.sum(axis=1, keepdims=True)
    cm_norm = np.nan_to_num(cm_norm)

short_names = [n[:14] for n in class_names]
fig, ax = plt.subplots(figsize=(12, 10))
fig.patch.set_facecolor('#FAFAFA')
im = ax.imshow(cm_norm, cmap='Purples', vmin=0, vmax=1)
ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(short_names, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(short_names, fontsize=9)
ax.set_xlabel('Predicción', fontsize=12, color=COLOR_DARK)
ax.set_ylabel('Especie real', fontsize=12, color=COLOR_DARK)
ax.set_title('Matriz de Confusión Normalizada — ATTENTION\n(Diagonal = Recall)', fontsize=13, color=COLOR_DARK)

for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        val = cm_norm[i, j]
        if val >= 0.01:
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    color='white' if val > 0.5 else COLOR_DARK, fontsize=8)

fig.colorbar(im, ax=ax, label='Proporción', fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig(RUN_DIR / 'confusion_matrix_normalized.png', dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
plt.show()

print('\nCONFUSIONES MÁS FUERTES (Attention):\n')
confusiones = []
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if i != j and cm_norm[i, j] > 0:
            confusiones.append((cm_norm[i, j], class_names[i], class_names[j], cm[i, j]))
confusiones.sort(reverse=True)
for prop, real, pred, count in confusiones[:12]:
    print(f'  {prop:5.1%} de "{real}" -> "{pred}"  ({count} clips)')

## Sección 3 — F1 vs cantidad de datos

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

f1_per_class = f1_score(y_true, y_pred, labels=range(NUM_CLASSES), average=None, zero_division=0)
support_test = np.array([(y_true == i).sum() for i in range(NUM_CLASSES)])

archivos_por_clase = {
    'no_ave': 1600, 'Celeus_grammicus': 28, 'Chordeiles_pusillus': 21,
    'Crypturellus_cinereus': 48, 'Crypturellus_undulatus': 29,
    'Frederickena_fulva': 20, 'Glaucidium_brasilianum': 22,
    'Lipaugus_vociferans': 36, 'Ramphastos_tucanus': 31,
    'Rupornis_magnirostris': 20, 'Trogon_viridis': 79,
}
n_archivos = np.array([archivos_por_clase[n] for n in class_names])
mask_aves = np.array([n != 'no_ave' for n in class_names])

corr_con_noave = np.corrcoef(n_archivos, f1_per_class)[0, 1]
corr_sin_noave = np.corrcoef(n_archivos[mask_aves], f1_per_class[mask_aves])[0, 1]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.patch.set_facecolor('#FAFAFA')
axes[0].scatter(n_archivos, f1_per_class, s=120, c=COLOR_PRIMARY, alpha=0.7, edgecolors=COLOR_DARK)
for i in range(NUM_CLASSES):
    axes[0].annotate(class_names[i][:10], (n_archivos[i], f1_per_class[i]),
                     fontsize=7, alpha=0.8, xytext=(5, 5), textcoords='offset points')
axes[0].set_xscale('log')
axes[0].set_xlabel('Archivos train (log)'); axes[0].set_ylabel('F1')
axes[0].set_title(f'Attention - Todas (r = {corr_con_noave:.3f})', color=COLOR_DARK)
axes[0].grid(alpha=0.3)

axes[1].scatter(n_archivos[mask_aves], f1_per_class[mask_aves], s=120, c=COLOR_ACCENT, alpha=0.7, edgecolors=COLOR_DARK)
for i in np.where(mask_aves)[0]:
    axes[1].annotate(class_names[i][:10], (n_archivos[i], f1_per_class[i]),
                     fontsize=7, alpha=0.8, xytext=(5, 5), textcoords='offset points')
z = np.polyfit(n_archivos[mask_aves], f1_per_class[mask_aves], 1)
xs = np.linspace(n_archivos[mask_aves].min(), n_archivos[mask_aves].max(), 50)
axes[1].plot(xs, np.poly1d(z)(xs), '--', color=COLOR_WARN, lw=2)
axes[1].set_xlabel('Archivos train'); axes[1].set_ylabel('F1')
axes[1].set_title(f'Attention - Solo aves (r = {corr_sin_noave:.3f})', color=COLOR_DARK)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RUN_DIR / 'f1_vs_datos.png', dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
plt.show()
print(f'Pearson r (todas):    {corr_con_noave:.3f}')
print(f'Pearson r (solo aves): {corr_sin_noave:.3f}')

## Sección 4 — Errores de alta confianza

In [ ]:
errores_mask = (y_true != y_pred)
idx_errores = np.where(errores_mask)[0]
idx_errores_ordenados = idx_errores[np.argsort(-y_conf[idx_errores])]

print(f'Total errores: {errores_mask.sum()} de {len(y_true)} ({errores_mask.mean():.1%})\n')
print('TOP 15 ERRORES DE ALTA CONFIANZA — ATTENTION:\n')
for idx in idx_errores_ordenados[:15]:
    print(f'  {y_conf[idx]:>6.1%}  {class_names[y_true[idx]]:<24} -> {class_names[y_pred[idx]]}')

# Histograma
fig, ax = plt.subplots(figsize=(11, 6))
fig.patch.set_facecolor('#FAFAFA')
conf_aciertos = y_conf[~errores_mask]
conf_errores = y_conf[errores_mask]
bins = np.linspace(0, 1, 30)
ax.hist(conf_aciertos, bins=bins, alpha=0.6, label=f'Aciertos (n={len(conf_aciertos)})', color=COLOR_ACCENT)
ax.hist(conf_errores, bins=bins, alpha=0.6, label=f'Errores (n={len(conf_errores)})', color=COLOR_WARN)
ax.set_xlabel('Confianza'); ax.set_ylabel('Clips')
ax.set_title('Distribución de confianza — ATTENTION', color=COLOR_DARK)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RUN_DIR / 'distribucion_confianza.png', dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
plt.show()

print(f'\nConfianza media aciertos: {conf_aciertos.mean():.3f}')
print(f'Confianza media errores:  {conf_errores.mean():.3f}')

## Sección 5 — Interpretación bioacústica de confusiones

In [ ]:
def genero(nombre):
    if nombre == 'no_ave': return 'no_ave'
    return nombre.split('_')[0]

print('CLASIFICACIÓN DE CONFUSIONES — ATTENTION:\n')
tipos_count = {'MISMO_GENERO': 0, 'FUGA_NOAVE': 0, 'INTER_GENERO': 0}
for prop, real, pred, count in confusiones[:12]:
    if pred == 'no_ave':
        tipo = '[FUGA A no_ave]'
        tipos_count['FUGA_NOAVE'] += 1
    elif genero(real) == genero(pred):
        tipo = '[MISMO GÉNERO]'
        tipos_count['MISMO_GENERO'] += 1
    else:
        tipo = '[INTER-GÉNERO]'
        tipos_count['INTER_GENERO'] += 1
    print(f'  {prop:5.1%}  {real} -> {pred}  {tipo}')

print(f'\nResumen tipos: {tipos_count}')

## Sección 6 — Resumen del análisis (Attention)

In [ ]:
from sklearn.metrics import f1_score as _f1

macro_f1 = _f1(y_true, y_pred, average='macro', zero_division=0)
weighted_f1 = _f1(y_true, y_pred, average='weighted', zero_division=0)
acc_global = (y_true == y_pred).mean()
noave_idx = label_map['no_ave']
errores_a_noave = ((y_true != y_pred) & (y_pred == noave_idx)).sum()
total_errores = (y_true != y_pred).sum()

resumen = f"""
{'=' * 70}
RESUMEN ANALISIS DE ERRORES — ATTENTION MODEL
{'=' * 70}

METRICAS GLOBALES
  Accuracy global         : {acc_global:.4f}  (Baseline: 0.6322)
  Macro F1                : {macro_f1:.4f}
  Weighted F1             : {weighted_f1:.4f}

PATRON DE ERRORES
  Total errores           : {total_errores} de {len(y_true)} ({total_errores/len(y_true):.1%})
  Errores a no_ave        : {errores_a_noave} ({errores_a_noave/total_errores:.1%} de errores)
  Conf. media aciertos    : {conf_aciertos.mean():.3f}
  Conf. media errores     : {conf_errores.mean():.3f}

CORRELACION DATOS-F1
  Pearson r (todas)       : {corr_con_noave:.3f}
  Pearson r (solo aves)   : {corr_sin_noave:.3f}

TIPOS DE CONFUSIONES (top 12)
  Mismo genero            : {tipos_count['MISMO_GENERO']}
  Fuga a no_ave           : {tipos_count['FUGA_NOAVE']}
  Inter-genero            : {tipos_count['INTER_GENERO']}

(La comparacion detallada con baseline esta en el notebook 09)
{'=' * 70}
"""
print(resumen)
with open(RUN_DIR / 'analisis_errores_resumen.txt', 'w', encoding='utf-8') as f:
    f.write(resumen)
print(f'Guardado en {RUN_DIR.name}/analisis_errores_resumen.txt')